In [ ]:
# F4-Candidate Fusion V1 | C1b x C5
# Kaggle CPU. Mount the six original assets listed in RUN.md and Run All.
import base64
import gzip
import hashlib
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys
import tempfile

CONFIG = {
    "input_root": "/kaggle/input",
    "output_root": "/kaggle/working/F4_Candidate_Fusion_V1",
    "s0_receipt": "",
    "s0_original_submission": "",
    "s0_archive_replay": "",
    "s0_oof_cache_dirs": [],
}
PAYLOAD_SHA256 = '3c7320295b58d0ee06d44ad5bc03f64e59321a1dfadd4cd7f74d29f17e21177e'
PACKED_SOURCE = '...'

dependencies = {"pycocotools": "pycocotools==2.0.11", "skimage": "scikit-image", "sklearn": "scikit-learn"}
missing = [package for module, package in dependencies.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
source = gzip.decompress(base64.b64decode(PACKED_SOURCE))
if hashlib.sha256(source).hexdigest() != PAYLOAD_SHA256:
    raise ValueError("Notebook source checksum differs")
env = os.environ.copy()
env.update(OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", PYTHONHASHSEED="0")
with tempfile.TemporaryDirectory(prefix="f4_candidate_") as folder:
    for name, text in json.loads(source).items():
        (Path(folder) / (name if "." in name else f"{name}.py")).write_bytes(text.encode("utf-8"))
    entry = "import json, sys; from run_fusion import run; run(json.loads(sys.argv[1]), sys.argv[2])"
    cmd = [sys.executable, "-u", "-c", entry, json.dumps(CONFIG), PAYLOAD_SHA256]
    with subprocess.Popen(cmd, cwd=folder, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace") as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
        if proc.wait():
            raise subprocess.CalledProcessError(proc.returncode, cmd[:2])